In [17]:
from pathlib import Path
import pandas as pd

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data/processed/model_data.csv").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Cannot find data/processed/model_data.csv")

DATA_FILE = ROOT / "data/processed/model_data.csv"

model_data = pd.read_csv(
    DATA_FILE,
    parse_dates=["Date", "LabelEndDate"],
)

print("Loaded:", DATA_FILE)
print("Rows:", len(model_data))
display(model_data.head())
display(model_data["Split"].value_counts())

Loaded: /Users/chenxinhui/financial-engineering-club-fall26/data/processed/model_data.csv
Rows: 1183162


,Date,Ticker,Split,LabelEndDate,ret_1d,mom_5d,mom_20d,mom_60d,vol_5d,vol_20d,volume_ratio_20d,range_pct,intraday_ret,target,target_rank
0,2016-03-31,A,train,2016-04-01,-0.006730,0.009628,0.020748,-0.020644,0.008600,0.014850,2.044486,0.011041,-0.003999,0.014471,0.796095
1,2016-03-31,AAPL,train,2016-04-01,-0.005203,0.026948,0.081786,0.040165,0.014012,0.009942,0.781569,0.009359,-0.006653,0.009175,0.605206
2,2016-03-31,ABBV,train,2016-04-01,0.002105,0.013485,0.025494,0.001908,0.008175,0.015902,0.776470,0.011204,0.001754,0.005252,0.433839
3,2016-03-31,ABT,train,2016-04-01,0.004322,0.025748,0.063295,-0.019474,0.005612,0.010441,0.800988,0.011714,0.004322,0.008606,0.585683
4,2016-03-31,ACGL,train,2016-04-01,-0.005177,0.008368,0.028200,0.032680,0.012670,0.007784,1.365329,0.009986,-0.005455,0.014768,0.815618


Split
train         810586
test          249133
validation    122459
purged           984
Name: count, dtype: int64

准备训练、验证和测试数据

本节将已经清洗好的 `model_data` 按照 `Split` 列分成训练集、验证集和测试集。

- `train`：用于训练 XGBoost 模型；
- `validation`：用于检查模型表现和选择参数；
- `test`：只保留已经知道下一交易日收益率的数据，用于最终评价；
- `test_signals`：保留所有测试期内可以生成预测的记录，包括暂时没有真实收益率的记录。

模型使用项目预先定义的 9 个特征：

- `ret_1d`
- `mom_5d`
- `mom_20d`
- `mom_60d`
- `vol_5d`
- `vol_20d`
- `volume_ratio_20d`
- `range_pct`
- `intraday_ret`

其中：

- `X` 表示模型输入的 9 个特征；
- `y` 表示模型要预测的下一交易日收益率 `target`。

最后，代码检查所有输入特征都没有缺失值，确保数据可以安全地用于模型训练。

In [18]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data.preparation import FEATURES

features = FEATURES.copy()

required_columns = ["Date", "Ticker", "Split", "target", *features]
missing_columns = sorted(set(required_columns) - set(model_data.columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

train = (
    model_data.loc[model_data["Split"].eq("train")]
    .dropna(subset=["target"])
    .copy()
)

validation = (
    model_data.loc[model_data["Split"].eq("validation")]
    .dropna(subset=["target"])
    .copy()
)

test_signals = (
    model_data.loc[model_data["Split"].eq("test")]
    .copy()
)

test = test_signals.dropna(subset=["target"]).copy()

X_train, y_train = train[features], train["target"]
X_validation, y_validation = validation[features], validation["target"]
X_test, y_test = test[features], test["target"]
X_test_signals = test_signals[features]

assert X_train.notna().all().all()
assert X_validation.notna().all().all()
assert X_test.notna().all().all()
assert X_test_signals.notna().all().all()

print("Features:", features)
print("Training rows:", len(train))
print("Validation rows:", len(validation))
print("Test rows with known returns:", len(test))
print("Test rows available for prediction:", len(test_signals))

Features: ['ret_1d', 'mom_5d', 'mom_20d', 'mom_60d', 'vol_5d', 'vol_20d', 'volume_ratio_20d', 'range_pct', 'intraday_ret']
Training rows: 810568
Validation rows: 122459
Test rows with known returns: 248635
Test rows available for prediction: 249133


训练XGBoost

In [19]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=100,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=False,
)

print("XGBoost training completed.")
print("Number of trees:", xgb_model.n_estimators)

XGBoost training completed.
Number of trees: 100


查看验证集的RMSE

In [20]:
import numpy as np

validation_predictions = xgb_model.predict(X_validation)

validation_rmse = np.sqrt(
    np.mean((validation_predictions - y_validation) ** 2)
)

print("Validation RMSE:", validation_rmse)

eval_history = xgb_model.evals_result()
print("Last recorded validation RMSE:",
      eval_history["validation_0"]["rmse"][-1])

Validation RMSE: 0.019119930077604715
Last recorded validation RMSE: 0.01911993005125029


简单基准比较

In [21]:
baseline_prediction = y_train.mean()

baseline_rmse = np.sqrt(
    np.mean((y_validation - baseline_prediction) ** 2)
)

print("XGBoost validation RMSE:", validation_rmse)
print("Mean-return baseline RMSE:", baseline_rmse)
print(
    "Relative improvement:",
    1 - validation_rmse / baseline_rmse
)

XGBoost validation RMSE: 0.019119930077604715
Mean-return baseline RMSE: 0.019071793966928412
Relative improvement: -0.0025239424649707853


每天的Spearman排名相关

In [22]:
validation_scored = validation[["Date", "Ticker", "target"]].copy()
validation_scored["prediction"] = validation_predictions

def daily_spearman(group):
    if group["prediction"].nunique() < 2:
        return np.nan
    if group["target"].nunique() < 2:
        return np.nan
    return group["prediction"].corr(
        group["target"],
        method="spearman",
    )

daily_scores = validation_scored.groupby("Date").apply(daily_spearman)

print("Scored validation dates:", daily_scores.notna().sum())
print("Undefined validation dates:", daily_scores.isna().sum())
print("Mean daily Spearman:", daily_scores.mean())
print("Median daily Spearman:", daily_scores.median())

Scored validation dates: 249
Undefined validation dates: 0
Mean daily Spearman: 0.008478711979102369
Median daily Spearman: -0.005348148310748893


/var/folders/8g/cxhpp2fj2wzfzyql094rfcnr0000gn/T/ipykernel_27837/26455632.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_scores = validation_scored.groupby("Date").apply(daily_spearman)


（保守版）XGBoost配置

In [23]:
tuned_xgb = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.03,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=10.0,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=50,
)

tuned_xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    verbose=False,
)

print("Best iteration:", tuned_xgb.best_iteration)
print("Best validation RMSE:", tuned_xgb.best_score)

Best iteration: 11
Best validation RMSE: 0.01907014159872521


调参后的Spearman排名

In [24]:
tuned_validation_predictions = tuned_xgb.predict(X_validation)

tuned_validation_rmse = np.sqrt(
    np.mean((tuned_validation_predictions - y_validation) ** 2)
)

tuned_scored = validation[["Date", "Ticker", "target"]].copy()
tuned_scored["prediction"] = tuned_validation_predictions

tuned_daily_scores = tuned_scored.groupby("Date").apply(daily_spearman)

print("Tuned validation RMSE:", tuned_validation_rmse)
print("Scored validation dates:", tuned_daily_scores.notna().sum())
print("Mean daily Spearman:", tuned_daily_scores.mean())
print("Median daily Spearman:", tuned_daily_scores.median())

Tuned validation RMSE: 0.019070141616494216
Scored validation dates: 249
Mean daily Spearman: 0.0034273583877810973
Median daily Spearman: 0.003974273395477181


/var/folders/8g/cxhpp2fj2wzfzyql094rfcnr0000gn/T/ipykernel_27837/285917067.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tuned_daily_scores = tuned_scored.groupby("Date").apply(daily_spearman)


对所有股票生成预测并保存为CSV

In [25]:
selected_model = xgb_model

test_predictions = selected_model.predict(X_test_signals)

prediction_output = test_signals[["Date", "Ticker"]].copy()
prediction_output["prediction"] = test_predictions

prediction_output = prediction_output.sort_values(
    ["Date", "Ticker"]
).reset_index(drop=True)

assert not prediction_output.duplicated(["Date", "Ticker"]).any()
assert np.isfinite(prediction_output["prediction"]).all()

PREDICTION_FILE = ROOT / "reports" / "xgboost_predictions.csv"
PREDICTION_FILE.parent.mkdir(parents=True, exist_ok=True)

prediction_output.to_csv(PREDICTION_FILE, index=False)

print("Saved:", PREDICTION_FILE)
print("Prediction rows:", len(prediction_output))
display(prediction_output.head())

Saved: /Users/chenxinhui/financial-engineering-club-fall26/reports/xgboost_predictions.csv
Prediction rows: 249133


,Date,Ticker,prediction
0,2024-01-02,A,0.000454
1,2024-01-02,AAPL,-0.000755
2,2024-01-02,ABBV,0.000184
3,2024-01-02,ABNB,0.000666
4,2024-01-02,ABT,0.000520


在模型未使用的测试集上生成的排名评价

In [26]:
test_predictions_known = xgb_model.predict(X_test)

test_scored = test[["Date", "Ticker", "target"]].copy()
test_scored["prediction"] = test_predictions_known

test_daily_scores = (
    test_scored
    .groupby("Date")[["prediction", "target"]]
    .apply(daily_spearman)
)

print("Scored test dates:", test_daily_scores.notna().sum())
print("Undefined test dates:", test_daily_scores.isna().sum())
print("Mean daily Spearman:", test_daily_scores.mean())
print("Median daily Spearman:", test_daily_scores.median())

Scored test dates: 501
Undefined test dates: 0
Mean daily Spearman: 0.0029928635903255993
Median daily Spearman: -0.0068266944731261376


改进（LambdaMART）

In [28]:
import numpy as np
import xgboost as xgb

y_train_relevance = (
    np.floor(train["target_rank"].to_numpy() * 5)
    .clip(0, 4)
    .astype(np.int32)
)

y_validation_relevance = (
    np.floor(validation["target_rank"].to_numpy() * 5)
    .clip(0, 4)
    .astype(np.int32)
)

qid_train = train["Date"].factorize(sort=True)[0]
qid_validation = validation["Date"].factorize(sort=True)[0]

assert np.isfinite(y_train_relevance).all()
assert np.isfinite(y_validation_relevance).all()
assert len(qid_train) == len(X_train)
assert len(qid_validation) == len(X_validation)

print("Training relevance labels:", np.unique(y_train_relevance))
print("Validation relevance labels:", np.unique(y_validation_relevance))

lambdamart_model = xgb.XGBRanker(
    objective="rank:ndcg",
    eval_metric="ndcg@10",
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

lambdamart_model.fit(
    X_train,
    y_train_relevance,
    qid=qid_train,
    verbose=False,
)

print("LambdaMART training completed.")

Training relevance labels: [0 1 2 3 4]
Validation relevance labels: [0 1 2 3 4]
LambdaMART training completed.


在验证集检查top10和spearman

In [29]:
from sklearn.metrics import ndcg_score

validation_rank_predictions = lambdamart_model.predict(X_validation)

validation_ranked = validation[
    ["Date", "Ticker", "target"]
].copy()

validation_ranked["prediction"] = validation_rank_predictions
validation_ranked["relevance"] = y_validation_relevance

def daily_ndcg(group):
    true_relevance = group["relevance"].to_numpy().reshape(1, -1)
    predicted_score = group["prediction"].to_numpy().reshape(1, -1)
    k = min(10, len(group))

    return ndcg_score(
        true_relevance,
        predicted_score,
        k=k,
    )

def daily_spearman(group):
    if group["prediction"].nunique() < 2:
        return np.nan
    if group["target"].nunique() < 2:
        return np.nan

    return group["prediction"].corr(
        group["target"],
        method="spearman",
    )

validation_ndcg = (
    validation_ranked
    .groupby("Date")
    .apply(daily_ndcg)
)

validation_spearman = (
    validation_ranked
    .groupby("Date")[["prediction", "target"]]
    .apply(daily_spearman)
)

print("Validation dates:", validation_ndcg.notna().sum())
print("Mean NDCG@10:", validation_ndcg.mean())
print("Median NDCG@10:", validation_ndcg.median())
print("Mean daily Spearman:", validation_spearman.mean())
print("Median daily Spearman:", validation_spearman.median())

Validation dates: 249
Mean NDCG@10: 0.5128470443226616
Median NDCG@10: 0.5099115580148588
Mean daily Spearman: 0.024168744074240703
Median daily Spearman: 0.03717930820142095


/var/folders/8g/cxhpp2fj2wzfzyql094rfcnr0000gn/T/ipykernel_27837/1759825439.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(daily_ndcg)


In [30]:
validation_sorted = validation_ranked.sort_values(
    ["Date", "prediction", "Ticker"],
    ascending=[True, False, True],
).reset_index(drop=True)

validation_sorted["prediction_rank"] = (
    validation_sorted.groupby("Date").cumcount() + 1
)

validation_top10 = validation_sorted.loc[
    validation_sorted["prediction_rank"] <= 10
]

daily_top10_return = (
    validation_top10
    .groupby("Date")["target"]
    .mean()
)

daily_all_return = (
    validation_ranked
    .groupby("Date")["target"]
    .mean()
)

print("Mean daily Top-10 return:", daily_top10_return.mean())
print("Mean daily all-stock return:", daily_all_return.mean())
print(
    "Top-10 advantage:",
    daily_top10_return.mean() - daily_all_return.mean(),
)

Mean daily Top-10 return: 0.0031377138459860836
Mean daily all-stock return: 0.0008882538206119635
Top-10 advantage: 0.0022494600253741203


In [31]:
test_rank_predictions = lambdamart_model.predict(X_test)

test_ranked = test[[
    "Date",
    "Ticker",
    "target",
    "target_rank",
]].copy()

test_ranked["prediction"] = test_rank_predictions
test_ranked["relevance"] = (
    np.floor(test_ranked["target_rank"] * 5)
    .clip(0, 4)
    .astype(np.int32)
)

test_ndcg = (
    test_ranked
    .groupby("Date")
    .apply(daily_ndcg, include_groups=False)
)

test_spearman = (
    test_ranked
    .groupby("Date")[["prediction", "target"]]
    .apply(daily_spearman)
)

test_sorted = test_ranked.sort_values(
    ["Date", "prediction", "Ticker"],
    ascending=[True, False, True],
).reset_index(drop=True)

test_sorted["prediction_rank"] = (
    test_sorted.groupby("Date").cumcount() + 1
)

test_top10 = test_sorted.loc[
    test_sorted["prediction_rank"] <= 10
]

test_daily_top10 = (
    test_top10
    .groupby("Date")["target"]
    .mean()
)

test_daily_all = (
    test_ranked
    .groupby("Date")["target"]
    .mean()
)

print("Mean test NDCG@10:", test_ndcg.mean())
print("Median test NDCG@10:", test_ndcg.median())
print("Mean test Spearman:", test_spearman.mean())
print("Median test Spearman:", test_spearman.median())
print("Mean test Top-10 return:", test_daily_top10.mean())
print("Mean test all-stock return:", test_daily_all.mean())
print(
    "Test Top-10 advantage:",
    test_daily_top10.mean() - test_daily_all.mean(),
)

Mean test NDCG@10: 0.5164581204436811
Median test NDCG@10: 0.5216611898816887
Mean test Spearman: 0.014378337279298608
Median test Spearman: 0.026287047949892
Mean test Top-10 return: 0.00294320346193873
Mean test all-stock return: 0.0007481087800461388
Test Top-10 advantage: 0.0021950946818925913


In [32]:
lambdamart_test_predictions = lambdamart_model.predict(
    X_test_signals
)

lambdamart_prediction_output = test_signals[
    ["Date", "Ticker"]
].copy()

lambdamart_prediction_output["prediction"] = (
    lambdamart_test_predictions
)

lambdamart_prediction_output = (
    lambdamart_prediction_output
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

assert not lambdamart_prediction_output.duplicated(
    ["Date", "Ticker"]
).any()
assert np.isfinite(
    lambdamart_prediction_output["prediction"]
).all()

LAMBDA_PREDICTION_FILE = (
    ROOT / "reports" / "lambdamart_predictions.csv"
)

LAMBDA_PREDICTION_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

lambdamart_prediction_output.to_csv(
    LAMBDA_PREDICTION_FILE,
    index=False,
)

print("Saved:", LAMBDA_PREDICTION_FILE)
print(
    "Prediction rows:",
    len(lambdamart_prediction_output),
)

display(lambdamart_prediction_output.head())

Saved: /Users/chenxinhui/financial-engineering-club-fall26/reports/lambdamart_predictions.csv
Prediction rows: 249133


,Date,Ticker,prediction
0,2024-01-02,A,-0.237463
1,2024-01-02,AAPL,-0.135531
2,2024-01-02,ABBV,-0.092277
3,2024-01-02,ABNB,-0.230522
4,2024-01-02,ABT,-0.342587


In [33]:
LAMBDA_FILE = (
    ROOT / "reports" / "lambdamart_predictions.csv"
)

lambdamart_predictions = pd.read_csv(
    LAMBDA_FILE,
    parse_dates=["Date"],
)

print(
    "LambdaMART prediction rows:",
    len(lambdamart_predictions),
)

display(lambdamart_predictions.head())

LambdaMART prediction rows: 249133


,Date,Ticker,prediction
0,2024-01-02,A,-0.237463
1,2024-01-02,AAPL,-0.135531
2,2024-01-02,ABBV,-0.092277
3,2024-01-02,ABNB,-0.230522
4,2024-01-02,ABT,-0.342587
